# RTrader

This notebook implements the grid trading strategy via [RoboMarkets API](https://api-doc.stockstrader.com/). This version is present in the repository as an interface with notebook environment to develop additional functionality. If you wish to use the strategy without making any major changes in the logic, I recommend to refer to the `scripts` folder of this repository.

---
# Contents

### 1) [Preparing Environment](#preparation)
### 2) [RTrader class](#api_client)
### 3) [GridStrategy class](#strategy)
### 4) [Sanity Check](#sanity)
### 5) [Conclusion](#results)


# 1) Preparing environment <a name="preparation"></a>

Please ensure that all packages are installed

In [224]:
import requests
import json
import time
import logging
from typing import Dict, List, Optional, Any
from datetime import datetime
import os

In [9]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('trading.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# 2) RTrader class <a name="api_client"></a>

The following class implements a client for the RoboMarkets. It should be noted that change of broker will likely result in errors as the structure of the schema may vary significantly, e.g. your account number may be not an endpoint but a header. I recommend familiarizing yourself with your broker's schema to ensure stable work.

In [242]:
class RTrader:
    def __init__(self, api_key: str, account_id: str,
                 base_url: str = 'https://api.stockstrader.com/api/v1/'):
        """
        API client class with basic functionality

        param api_key: personal API key
        param account_id: id of target account
        param base_url: prefix url for all endpoints
        """
        
        self.api_key = api_key
        self.account_id = account_id
        self.base_url = base_url
        self.headers = {'Authorization': f'Bearer {api_key}',
                        'Content-Type': "application/json",
                        'Accept': "application/json"}

        
    def _make_request(self, method: str, endpoint: str, extra_headers: Dict = {}, data: Dict = {}):
        """
        Make request to the server

        param method: GET, POST or DELETE
        param endpoint: target endpoint of the schema
        param extra_headers: additional headers to add to the request
        param data: necessary data for the request

        :returns: json of the response
        """
        url = f'{self.base_url}{endpoint}'

        try:
            if method.upper() == 'GET':
                response = requests.get(url, headers=self.headers | extra_headers, timeout=10)
            elif method.upper() == 'POST':
                response = requests.post(url, headers=self.headers | extra_headers, data=data, timeout=10)
            elif method.upper() == 'DELETE':
                response = requests.delete(url, headers=self.headers | extra_headers, timeout=10)
            else:
                logger.error(f"Unsupported HTTP method: {method}; currently only GET, POST, DELETE implemented")
                return None
                
            response.raise_for_status()
            return response.json() if response.content else None
            
        except requests.exceptions.RequestException as e:
            logger.error(f"API request failed: {e}")
            if hasattr(e, 'response') and e.response:
                logger.error(f"Response: {e.response.text}")
            return None


    def get_orders(self, status: str = 'active') -> List[Dict]:
        """
        Receive list of orders
        
        param sta
        :returns: list of all active limit orders
        """
        logger.info("Fetching open orders...")
        response = self._make_request("GET", f"accounts/{self.account_id}/orders", extra_headers={'status': status})
        
        if response and 'data' in response:
            
            # Filter for limit orders only
            limit_orders = [order for order in response['data'] if order['type'] == 'limit']
            logger.info(f"Found {len(limit_orders)} open limit orders")
            
            return limit_orders
            
        return []

    
    def place_order(self, order_params: Dict) -> Optional[Dict]:
        """
        Place an order

        param order_params: a dict of order parameters
        :returns: None if required parametrs missing else responce from _make_request
        """
        # Required parameters
        required_fields = ['ticker', 'side', 'type', 'volume', 'price']
        
        for field in required_fields:
            if field not in order_params.keys():
                logger.error(f"Missing required order parameter: {field}")
                return None
        
        # Prepare order structure
        order_data = {
            'ticker': order_params['ticker'],
            'side': order_params['side'],
            'type': order_params['type'],
            'price': order_params['price'],
            'volume': str(order_params['volume'])
        }
        
        # Add optional parameters
        if 'take_profit' in order_params:
            order_data['take_profit'] = str(order_params['take_profit'])
        if 'stop_loss' in order_params:
            order_data['stop_loss'] = str(order_params['stop_loss'])
            
        logger.info(f"Placing limit order: {order_data['side']} {order_data['volume']} "
                   f"{order_data['ticker']} @ {order_data['price']}")
        
        response = self._make_request('POST', f"accounts/{self.account_id}/orders", data=order_data)
        
        if response:
            logger.info(f"Order placed successfully: {response.get('data', 'unknown').get('order_id', 'unknown')}")
        else:
            logger.error("Failed to place order")
            
        return response

    
    def cancel_order(self, order_id: str) -> bool:
        """
        Cancel order

        param order_id: id of the order to cancel
        :returns: True on success, False on failure
        """
        
        logger.info(f"Cancelling order: {order_id}")
        response = self._make_request('DELETE', f"accounts/{self.account_id}/orders/{order_id}")
        
        if response is not None:
            logger.info(f"Order {order_id} cancelled successfully")
            return True
        else:
            logger.error(f"Failed to cancel order {order_id}")
            return False

# 3) Strategy class <a name="strategy"></a>
The following class implemets the "decision maker" class. As long as the functionality of the client is mainatained this class may work with any of the brokers.

In [269]:
class GridStrategy():
    def __init__(self, api_client: RTrader, grid_file: str = 'grid.txt'):
        """
        Grid based strategy controller for the trader

        param api_client: instance of the trader class
        param grid_file: path to the grid configuration
        """
        self.api_client = api_client
        self.grid_file = grid_file
        
        self.active_order_ids = []
        self.active_order_keys = []
        self.filled_order_keys = []

        self.grid_orders = self.load_grid()
        self.grid_keys = [self.get_order_key(order) for order in self.grid_orders]

    
    @staticmethod
    def get_order_key(order: Dict) -> str:
        """
        Create local key for order tracking
        
        param order: order information
        :returns: local order key
        """
        return f"{order['ticker']}_{order['side']}_{order['price']:.2f}"


    def load_grid(self) -> List[Dict]:
        """
        Load grid configuration from the specified file

        :returns: list of orders in configuration
        """
        if not os.path.exists(self.grid_file):
            logger.error(f"Grid file {self.grid_file} not found")
            return []
        
        orders = []
        
        try:
            with open(self.grid_file, 'r') as f:
                content = f.read().strip()
                
                #JSON format required
                orders = json.loads(content)
                        
            logger.info(f"Loaded {len(orders)} orders from {self.grid_file}")
            self.grid_orders = orders
            return orders
            
        except Exception as e:
            logger.error(f"Error loading grid file: {e}")
            return []


    def sync_state(self) -> bool:
        """
        Synchronize the server orders with local state

        :returns: True on success, False on failure
        """
        logger.info("Synchronizing strategy state with server")
        try:
            # Get current active a orders from the server
            active_orders = self.api_client.get_orders(status='active')
            filled_orders = self.api_client.get_orders(status='filled')
    
            # Write active ids and keys
            self.active_order_keys = [self.get_order_key(order) for order in active_orders]
            self.active_order_ids = [order['id'] for order in active_orders]
    
            # Write keys of filled orders
            self.filled_order_keys = [self.get_order_key(order) for order in filled_orders]
            
            return True

        except Exception as e:
            logger.error(f"`State synchronization error: {e}")
            return False
        

    def sync_orders(self) -> bool:
        """
        Synchronize the server orders with local grid configuration

        :returns: True on success, False on failure
        """
        logger.info("Synchronizing grid with server")
        
        if not self.grid_orders:
            logger.warning("No grid orders to process")
            return False

        if not self.sync_state():
            logger.error("Unable to syncronize local state")
            return False
        
        # Track which grid orders need to be placed
        orders_to_place = []
        
        for order in self.grid_orders:
            order_key = self.get_order_key(order)
            
            if order_key not in self.active_order_keys or order_key not in self.filled_order_keys:
                logger.info(f"Missing order detected: {order['ticker']} {order['side']} at {order['price']:.2f}")
                orders_to_place.append(order)
            else:
                logger.debug(f"Order already exists or has been filled: {order_key}")
        
        # Place missing orders
        if orders_to_place:
            logger.info(f"Placing {len(orders_to_place)} missing orders...")
            
            for order in orders_to_place:
                response = self.api_client.place_order(order)
                if response:
                    # Store order_id if returned
                    if 'order_id' in response['data']:
                        self.active_order_ids.append(response['data']['order_id'])
                else:
                    logger.error(f"Failed to place order: {order}")
                    
                # Small delay to avoid server side limits
                time.sleep(0.5)
        else:
            logger.info("All grid orders are already placed")
        
        logger.info("Grid synchronization completed")
        
        return True


    def remove_all_orders(self):
        """
        Remove all active orders based on local state

        :returns: True on success, False on failure
        """
        try:
            for order_id in self.active_order_ids:
                self.api_client.cancel_order(order_id)
            return True
            
        except Exception as e:
            logger.error(f"Error removing orders: {e}")
            return False
            

    def execute(self, check_interval: int = 30):
        """
        Execute strategy
        
        param check_interval: interval in seconds between checks
        """
        logger.info(f"Starting grid routine, interval: {check_interval} seconds)")
        
        try:
            while True:
                # Check for filled orders, add missing
                if not self.sync_orders():
                    logger.info("Unable to execute routine")
                    break
                
                # Wait before next check
                logger.info(f"Waiting {check_interval} seconds until next check...")
                time.sleep(check_interval)
                
        except KeyboardInterrupt:
            logger.info("Monitoring stopped by user")
        except Exception as e:
            logger.error(f"Monitoring loop error: {e}")

# 4) Sanity Check <a name='sanity'></a>

You can test the strategy in this section.

In [ ]:
API_KEY = "YOUR_API_KEY"
BASE_URL = "https://api.stockstrader.com/api/v1/"
ACCOUNT_ID = "YOUR_ACCOUNT_ID"

In [ ]:
def build_sample_grid():
    """
    Create an example grid.txt file with sample limit orders
    """
    example_orders = [
        {'ticker': 'NVDA.nq', 'side': 'buy', 'type': 'limit', 'volume': 10, 'price': 150.00, 'take_profit': 155.00, 'stop_loss': 148.00},
        {'ticker': 'NVDA.nq', 'side': 'buy', 'type': 'limit', 'volume': 10, 'price': 151.00, 'take_profit': 155.00, 'stop_loss': 148.00},
        {'ticker': 'NVDA.nq', 'side': 'buy', 'type': 'limit', 'volume': 10, 'price': 152.00, 'take_profit': 155.00, 'stop_loss': 148.00},
        {'ticker': 'NVDA.nq', 'side': 'buy', 'type': 'limit', 'volume': 10, 'price': 153.00, 'take_profit': 155.00, 'stop_loss': 148.00},
        {'ticker': 'NVDA.nq', 'side': 'buy', 'type': 'limit', 'volume': 10, 'price': 154.00, 'take_profit': 155.00, 'stop_loss': 148.00}
    ]
    
    with open('grid.txt', 'w') as f:
        json.dump(example_orders, f, indent=2)

In [271]:
# Initialize API client
client = RTrader(
    api_key=API_KEY,
    account_id=ACCOUNT_ID,
    base_url=BASE_URL
)
   
# Initialize grid strategy
strategy = GridStrategy(client, grid_file="grid.txt")

2026-04-18 14:28:18,657 - INFO - Loaded 5 orders from grid.txt


In [272]:
strategy.execute()

2026-04-18 14:28:24,239 - INFO - Starting grid routine, interval: 30 seconds)
2026-04-18 14:28:24,241 - INFO - Synchronizing grid with server
2026-04-18 14:28:24,244 - INFO - Synchronizing strategy state with server
2026-04-18 14:28:24,246 - INFO - Fetching open orders...
2026-04-18 14:28:24,454 - INFO - Found 0 open limit orders
2026-04-18 14:28:24,461 - INFO - Fetching open orders...
2026-04-18 14:28:24,596 - INFO - Found 0 open limit orders
2026-04-18 14:28:24,597 - INFO - Missing order detected: NVDA.nq buy at 150.00
2026-04-18 14:28:24,599 - INFO - Missing order detected: NVDA.nq buy at 151.00
2026-04-18 14:28:24,602 - INFO - Missing order detected: NVDA.nq buy at 152.00
2026-04-18 14:28:24,604 - INFO - Missing order detected: NVDA.nq buy at 153.00
2026-04-18 14:28:24,606 - INFO - Missing order detected: NVDA.nq buy at 154.00
2026-04-18 14:28:24,607 - INFO - Placing 5 missing orders...
2026-04-18 14:28:24,610 - INFO - Placing limit order: buy 10 NVDA.nq @ 150.0
2026-04-18 14:28:24

# 5) Conclusion <a name='results'></a>

While this project implementents one of the most basic trading algorithms with not the most impressive returns (avg. 2% per month), it nevertheless may serve as a foundation for more advanced trading. The RoboMarkets have their own spread on trading and does not provide LOB data via their API, so if you want to transform this project into an HFT version you will have to add additional methods both to store/train model and to fetch the current state of the Limit Order Book.

This algorithm works on the logic that after turning on user may simply forget about it during the entire trading day. It should also be noted that even this simple version may be improved quite easily by adding safety measures like not opening additional limit orders if the price has dropped significantly or other risk control methods, e.g. risk-based model.